# PY400 — Pandas DataFrames
### Official documentation: https://pandas.pydata.org/docs/

A **DataFrame** is a 2D labelled data structure — think of it as a spreadsheet or SQL table in Python. It is the central object in Pandas and the backbone of data analysis in Python.

## Topics Covered
1. Creating DataFrames
2. Selecting columns and rows
3. Filtering (WHERE in SQL)
4. Sorting
5. Grouping and aggregation (GROUP BY + HAVING)
6. Adding / removing columns
7. SQL comparison cheat sheet
8. NumPy vs Pandas — when to use which

> **Perspective:** If NumPy is the engine (fast numeric computation on arrays), Pandas is the car (structured, labelled data with rich operations). You don't choose one over the other — you use both, each for what it does best.

In [ ]:
## Creating DataFrames

import pandas as pd
import numpy as np

# From a dictionary (most common)
df = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hank'],
    'dept':       ['IT', 'HR', 'IT', 'Finance', 'IT', 'HR', 'Finance', 'IT'],
    'salary':     [70000, 55000, 85000, 60000, 90000, 48000, 75000, 95000],
    'experience': [5, 3, 8, 4, 10, 2, 6, 12],
    'rating':     [4.2, 3.8, 4.5, 4.0, 4.8, 3.5, 4.1, 4.9]
})

print("=== First look ===")
print(df)
print(f"\nShape : {df.shape}")      # (rows, columns)
print(f"Types :\n{df.dtypes}")

In [ ]:
## Essential inspection methods

print("=== .head() — first 5 rows ===")
print(df.head())

print("\n=== .info() — types, non-null counts, memory ===")
df.info()

print("\n=== .describe() — summary statistics ===")
print(df.describe())

print("\n=== .nunique() — unique values per column ===")
print(df.nunique())

---
## Selecting Columns and Rows

In [ ]:
## Column selection

# Single column → returns a Series
print(df['name'])

print("---")

# Multiple columns → returns a DataFrame
print(df[['name', 'salary', 'dept']])

# SQL equivalent: SELECT name, salary, dept FROM employees

In [ ]:
## Row selection — .loc (label) vs .iloc (integer position)

# .iloc — by integer index (like array indexing)
print("First row       :", df.iloc[0].tolist())
print("Rows 1-3        :")
print(df.iloc[1:4])

print("---")

# .loc — by label (index value) + column name
print("Row 0, name col :", df.loc[0, 'name'])
print("Rows 0-2, selected cols:")
print(df.loc[0:2, ['name', 'salary']])

# SQL equivalent: SELECT name, salary FROM employees LIMIT 3

---
## Filtering Rows (SQL WHERE)

In [ ]:
## Filtering — single and multiple conditions

# Single condition
high_earners = df[df['salary'] > 70000]
print("=== Salary > 70K ===")
print(high_earners)
# SQL: SELECT * FROM employees WHERE salary > 70000

print("---")

# Multiple conditions (use & for AND, | for OR, ~ for NOT)
it_seniors = df[(df['dept'] == 'IT') & (df['experience'] > 5)]
print("=== IT dept AND experience > 5 ===")
print(it_seniors)
# SQL: SELECT * FROM employees WHERE dept = 'IT' AND experience > 5

print("---")

# isin() — like SQL IN
selected_depts = df[df['dept'].isin(['IT', 'Finance'])]
print("=== Dept IN ('IT', 'Finance') ===")
print(selected_depts)
# SQL: SELECT * FROM employees WHERE dept IN ('IT', 'Finance')

print("---")

# String matching
a_names = df[df['name'].str.startswith('A')]
print("=== Name starts with 'A' ===")
print(a_names)
# SQL: SELECT * FROM employees WHERE name LIKE 'A%'

---
## Sorting (SQL ORDER BY)

In [ ]:
## Sorting

# Sort by salary descending
print(df.sort_values('salary', ascending=False))
# SQL: SELECT * FROM employees ORDER BY salary DESC

print("---")

# Sort by multiple columns
print(df.sort_values(['dept', 'salary'], ascending=[True, False]))
# SQL: SELECT * FROM employees ORDER BY dept ASC, salary DESC

---
## Grouping and Aggregation (SQL GROUP BY + HAVING)

In [ ]:
## GROUP BY equivalent

# Single aggregation
print("=== Avg salary by dept ===")
print(df.groupby('dept')['salary'].mean())
# SQL: SELECT dept, AVG(salary) FROM employees GROUP BY dept

print("---")

# Multiple aggregations
print("=== Multiple agg by dept ===")
print(df.groupby('dept').agg({
    'salary':     ['mean', 'max', 'min'],
    'experience': 'mean',
    'name':       'count'      # COUNT(*)
}))
# SQL: SELECT dept, AVG(salary), MAX(salary), MIN(salary), AVG(experience), COUNT(*)
#      FROM employees GROUP BY dept

print("---")

# HAVING equivalent — filter groups AFTER aggregation
dept_avg = df.groupby('dept')['salary'].mean()
print("=== Depts with avg salary > 60K (HAVING) ===")
print(dept_avg[dept_avg > 60000])
# SQL: SELECT dept, AVG(salary) FROM employees GROUP BY dept HAVING AVG(salary) > 60000

---
## Adding and Removing Columns

In [ ]:
## Adding and removing columns

df_copy = df.copy()

# Add a computed column
df_copy['salary_per_year_exp'] = df_copy['salary'] / df_copy['experience']

# Add a conditional column (like SQL CASE WHEN)
df_copy['seniority'] = np.where(df_copy['experience'] >= 7, 'Senior', 'Junior')

print(df_copy[['name', 'salary', 'experience', 'salary_per_year_exp', 'seniority']])

# Drop a column
df_copy = df_copy.drop(columns=['salary_per_year_exp'])
print(f"\nColumns after drop: {df_copy.columns.tolist()}")

---
## SQL vs Pandas Cheat Sheet

| SQL | Pandas |
|---|---|
| `SELECT col1, col2` | `df[['col1', 'col2']]` |
| `SELECT *` | `df` |
| `SELECT DISTINCT col` | `df['col'].unique()` |
| `WHERE col > 10` | `df[df['col'] > 10]` |
| `WHERE col IN (a, b)` | `df[df['col'].isin([a, b])]` |
| `WHERE col LIKE 'A%'` | `df[df['col'].str.startswith('A')]` |
| `ORDER BY col DESC` | `df.sort_values('col', ascending=False)` |
| `LIMIT 10` | `df.head(10)` |
| `GROUP BY col` | `df.groupby('col')` |
| `HAVING AVG(x) > 5` | `grouped[grouped > 5]` |
| `COUNT(*)` | `df.groupby('col').size()` |
| `JOIN` | `pd.merge(df1, df2, on='key')` |
| `UNION` | `pd.concat([df1, df2])` |
| `CASE WHEN` | `np.where(cond, true_val, false_val)` |
| `ALTER TABLE ADD` | `df['new_col'] = values` |

---
## NumPy vs Pandas — When to Use Which

| Criteria | NumPy | Pandas |
|---|---|---|
| **Data type** | Homogeneous arrays (all same type) | Heterogeneous tables (mixed types) |
| **Labels** | Integer indices only | Named columns and custom indices |
| **Missing data** | Manual (`np.nan` in float arrays) | Native support (`NaN`, `.fillna()`) |
| **Speed** | Faster for pure numeric math | Slower due to label overhead |
| **Use case** | Matrix math, linear algebra, images | CSV/Excel data, EDA, data wrangling |
| **SQL-like ops** | No | Yes (filter, group, join, pivot) |
| **Memory** | More efficient | Higher overhead per cell |

### Decision Rule
- **Use NumPy** when working with uniform numeric arrays (matrices, vectors, images, signals).
- **Use Pandas** when working with tabular data that has column names, mixed types, or missing values.
- **Use both** — Pandas DataFrames store numeric columns as NumPy arrays under the hood. You can access them via `.values` or `.to_numpy()`.

> **Perspective:** In practice, Pandas is your daily driver for data loading, cleaning, and exploration. NumPy takes over when you need raw computational speed inside algorithms (e.g., implementing gradient descent, matrix factorization).